# Part C — Support Ticket Triage with Guardrails

This notebook solves every Part C task with the supplied **offline** `mock_llm.py`. Run all cells from top to bottom after Part B has produced `outputs/customer_segments.csv`. It works when opened from either the repository root or `part_c/`; the project environment needs pandas and the dependencies in `requirements.txt`.

The experiment processes tickets in CSV order, never uses an API key or network service, and writes only text/code outputs. Model accuracy is measured on classified tickets; rejection coverage is reported separately so a high accuracy cannot hide human-review work.


In [1]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
ROOT = next((candidate for candidate in [HERE, *HERE.parents]
             if (candidate / "mock_llm.py").is_file() and (candidate / "part_c").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project root or part_c folder")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from collections import Counter
import json
import random
import pandas as pd
import mock_llm
from part_c.guardrails import (MAX_INPUT_TOKENS, EMAIL_REGEX, PHONE_REGEX, ORDER_REGEX,
    check_input_guardrails, redact_pii, verify_zero_pii_remaining)
from part_c.classifier import (ALLOWED_CATEGORIES, BASE_DELAY, ZERO_SHOT_PROMPT,
    FEW_SHOT_PROMPT, call_with_retry, validate_response)
from part_c.triage_pipeline import (OUTPUT_COLUMNS, CLASSIFIED_STATUSES,
    load_customer_segments, route_ticket, run_ticket_triage, summarise_run)
from part_c.run_part_c import audit_inputs, assert_full_run, compare_prompts, build_report

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
assert (OUTPUT_DIR / "customer_segments.csv").is_file(), "Run Part B first"
print("Repository:", ROOT.name)
print("Model:", "supplied deterministic offline mock")


Repository: Capstone _Project
Model: supplied deterministic offline mock


## 1. Load tickets and apply input guardrails before model calls

The threshold applies to **raw** text so that masking cannot turn a long ticket into an eligible one. Empty or whitespace-only tickets get `skipped_empty`; more than 400 estimated tokens gets `flagged_long`. Both statuses go to human review with no call and zero tokens. The original CSV order is kept throughout.


In [2]:
tickets = pd.read_csv(ROOT / "data/raw/tickets.csv", keep_default_na=False, dtype=str)
input_audit = audit_inputs(tickets)
guardrail_rows = []
for row in tickets.to_dict("records"):
    status, estimated = check_input_guardrails(row["text"])
    guardrail_rows.append({"ticket_id": row["ticket_id"], "input_status": status,
                           "estimated_tokens": estimated})
guardrails = pd.DataFrame(guardrail_rows)
assert len(tickets) == 164
assert MAX_INPUT_TOKENS == 400
assert input_audit["guardrail_counts"] == {"eligible": 156, "skipped_empty": 4, "flagged_long": 4}
assert guardrails.loc[guardrails["input_status"].eq("flagged_long"), "estimated_tokens"].eq(734).all()
assert check_input_guardrails("   ")[0] == "skipped_empty"
print("Input status counts:")
print(guardrails["input_status"].value_counts().to_string())
print("Long ticket estimates:")
print(guardrails[guardrails["input_status"].eq("flagged_long")].to_string(index=False))


Input status counts:
input_status
eligible         156
skipped_empty      4
flagged_long       4
Long ticket estimates:
ticket_id input_status  estimated_tokens
    T0163 flagged_long               734
    T0162 flagged_long               734
    T0164 flagged_long               734
    T0161 flagged_long               734


## 2. Redact PII in the exact required order

Email uses `[\w.+-]+@[\w-]+\.[\w.]+` → `[EMAIL]`; phone uses `\b[6-9]\d{9}\b` → `[PHONE]`; order ID uses `ORD-\d{6}` → `[ORDER_ID]`. Count replacements across **all 164 tickets**, including skipped and long tickets, then run the same patterns over the result. Only the masked text is allowed in the user message; neither the customer ID nor the order metadata is appended to it.


In [3]:
masked_texts = []
substitution_counts = Counter({"email": 0, "phone": 0, "order_id": 0, "total": 0})
remaining_counts = Counter({"email": 0, "phone": 0, "order_id": 0})
for text in tickets["text"]:
    # redact_pii implements email.subn, then phone.subn, then order.subn.
    masked, counts = redact_pii(text)
    masked_texts.append(masked)
    substitution_counts.update(counts)
    remaining_counts.update({"email": len(EMAIL_REGEX.findall(masked)),
                             "phone": len(PHONE_REGEX.findall(masked)),
                             "order_id": len(ORDER_REGEX.findall(masked))})
    assert verify_zero_pii_remaining(masked)
assert dict(substitution_counts) == {"email": 55, "phone": 91, "order_id": 132, "total": 278}
assert dict(remaining_counts) == {"email": 0, "phone": 0, "order_id": 0}
assert input_audit["pii_substitutions"] == dict(substitution_counts)
print("Ordered patterns:", EMAIL_REGEX.pattern, PHONE_REGEX.pattern, ORDER_REGEX.pattern)
print("Substitutions across all tickets:", dict(substitution_counts))
print("PII remaining after masking:", dict(remaining_counts))
print("A safely masked example:", masked_texts[0])


Ordered patterns: [\w.+-]+@[\w-]+\.[\w.]+ \b[6-9]\d{9}\b ORD-\d{6}
Substitutions across all tickets: {'email': 55, 'phone': 91, 'order_id': 132, 'total': 278}
PII remaining after masking: {'email': 0, 'phone': 0, 'order_id': 0}
A safely masked example: There is an unauthorized transaction on my card linked to order [ORDER_ID]. I did not place it. Call [PHONE].


## 3. Define zero-shot and few-shot system prompts

Both prompts say `CLASSIFY`, list billing, delivery, technical and fraud, and demand only `{"category": "...", "reason": "..."}`. The zero-shot prompt has no `EXAMPLES:` marker. The few-shot prompt has two labelled examples under that marker, including an account-takeover fraud example. Example order references are already placeholders.


In [4]:
for prompt in [ZERO_SHOT_PROMPT, FEW_SHOT_PROMPT]:
    assert "CLASSIFY" in prompt
    assert all(category in prompt for category in ALLOWED_CATEGORIES)
    assert '{"category": "...", "reason": "..."}' in prompt
    assert verify_zero_pii_remaining(prompt)
assert "EXAMPLES:" not in ZERO_SHOT_PROMPT
assert "\nEXAMPLES:\n" in FEW_SHOT_PROMPT
assert FEW_SHOT_PROMPT.count("Ticket:") >= 2 and '"category": "fraud"' in FEW_SHOT_PROMPT
print("Zero-shot system prompt:\n", ZERO_SHOT_PROMPT)
print("\nFew-shot system prompt:\n", FEW_SHOT_PROMPT)
print("Estimated system tokens:", {"zero_shot": mock_llm.est_tokens(ZERO_SHOT_PROMPT),
                                   "few_shot": mock_llm.est_tokens(FEW_SHOT_PROMPT)})


Zero-shot system prompt:
 CLASSIFY the support ticket into exactly one of: billing, delivery, technical, fraud.
You must respond strictly in valid JSON format:
{"category": "...", "reason": "..."}

Few-shot system prompt:
 CLASSIFY the support ticket into exactly one of: billing, delivery, technical, fraud.
You must respond strictly in valid JSON format:
{"category": "...", "reason": "..."}
EXAMPLES:
Ticket: I was charged twice for order [ORDER_ID]. Please refund the extra payment.
Response: {"category": "billing", "reason": "charged twice for order"}
Ticket: Someone hacked my account and made unauthorized transactions.
Response: {"category": "fraud", "reason": "unauthorized account takeover"}
Estimated system tokens: {'zero_shot': 32, 'few_shot': 81}


## 4. Retry transient errors with bounded exponential backoff

`call_with_retry` permits one initial attempt and at most three retries. Retry number `n` waits `0.05 × 2**n × uniform(0.9, 1.15)` seconds, with one log line showing ticket ID, number and wait. A local random generator seeded with 42 makes the waits reproducible without changing the global random state. The synthetic contract check below verifies retries 0, 1 and 2 and confirms a persistent failure stops after four attempts; its injected sleeper avoids unnecessary waiting. Full runs below use real waits and call `mock_llm.reset()` immediately before the batch.


In [5]:
original_chat = mock_llm.chat
attempts = {"count": 0}
def fail_three_then_succeed(messages):
    attempts["count"] += 1
    if attempts["count"] <= 3:
        raise mock_llm.TransientLLMError("temporary failure")
    return "recovered"
retry_contract_events = []
try:
    mock_llm.chat = fail_three_then_succeed
    result = call_with_retry([], "retry_contract", rng=random.Random(42),
        retry_events=retry_contract_events, sleeper=lambda seconds: None)
finally:
    mock_llm.chat = original_chat
assert result == "recovered" and attempts["count"] == 4
assert [event["retry_number"] for event in retry_contract_events] == [0, 1, 2]
for event in retry_contract_events:
    base = BASE_DELAY * 2 ** event["retry_number"]
    assert 0.9 * base <= event["wait_seconds"] <= 1.15 * base

attempts["count"] = 0
def always_transient(messages):
    attempts["count"] += 1
    raise mock_llm.TransientLLMError("persistent failure")
try:
    mock_llm.chat = always_transient
    try:
        call_with_retry([], "retry_limit", rng=random.Random(42), sleeper=lambda seconds: None)
    except RuntimeError:
        print("Persistent failure correctly stopped after four attempts")
    else:
        raise AssertionError("Expected the retry limit to stop")
finally:
    mock_llm.chat = original_chat
assert attempts["count"] == 4
print("Retry contract verified; no real model or network was used")


[RETRY] Ticket: retry_contract | Retry: 0 | Wait: 0.052993s
[RETRY] Ticket: retry_contract | Retry: 1 | Wait: 0.090625s
[RETRY] Ticket: retry_contract | Retry: 2 | Wait: 0.193751s
[RETRY] Ticket: retry_limit | Retry: 0 | Wait: 0.052993s
[RETRY] Ticket: retry_limit | Retry: 1 | Wait: 0.090625s
[RETRY] Ticket: retry_limit | Retry: 2 | Wait: 0.193751s
Persistent failure correctly stopped after four attempts
Retry contract verified; no real model or network was used


## 5. Validate every model answer without guessing

If JSON parsing succeeds, accept only a dictionary whose category is one of the four exact allowed values. A valid JSON response with an invalid category stays invalid even if its `reason` includes an allowed keyword. If parsing fails, extract lowercase alphabetic words with `re.findall(r"[a-z]+", text.lower())`; exactly one distinct allowed category yields `classified_keyword`. Empty, ambiguous or other outputs are flagged for a human.


In [6]:
validation_cases = [
    ('{"category": "billing", "reason": "charged twice"}', ("classified_json", "billing")),
    ("The customer appears to have a billing-related problem.", ("classified_keyword", "billing")),
    ('{"category": "shipping", "reason": "billing"}', ("flagged_invalid_output", None)),
    ('["billing"]', ("flagged_invalid_output", None)),
    ('"billing"', ("flagged_invalid_output", None)),
    ("billing and fraud", ("flagged_invalid_output", None)),
    ("no category mentioned", ("flagged_invalid_output", None)),
    (None, ("flagged_invalid_output", None)),
]
for content, expected in validation_cases:
    actual = validate_response(content)
    assert actual == expected, (content, actual, expected)
print("Validation edge cases passed:", len(validation_cases))


Validation edge cases passed: 8


## 6. Run zero-shot without a token budget and audit every submitted message

A spy wraps the supplied mock during this run to prove that **every attempt**, including retries, receives exactly the redacted eligible ticket text. The expected submission sequence is reconstructed from CSV order and retry IDs. This demonstrates that the four empty and four long tickets never produce calls. Truth labels are used only after classification for evaluation.


In [7]:
runs = {}
retry_logs = {}
def logger_for(name):
    retry_logs[name] = []
    def logger(line):
        retry_logs[name].append(line)
        print(line)
    return logger

submitted_user_texts = []
original_chat = mock_llm.chat
def audited_chat(messages, *args, **kwargs):
    assert [message["role"] for message in messages] == ["system", "user"]
    for message in messages:
        assert verify_zero_pii_remaining(message["content"])
    submitted_user_texts.append(messages[1]["content"])
    return original_chat(messages, *args, **kwargs)
try:
    mock_llm.chat = audited_chat
    runs["zero_shot"] = run_ticket_triage(prompt=ZERO_SHOT_PROMPT, token_budget=None,
        retry_logger=logger_for("zero_shot"))
finally:
    mock_llm.chat = original_chat
zero_df, zero_tokens = runs["zero_shot"]
assert_full_run(zero_df, correct=126)
retry_counts = Counter(event["ticket_id"] for event in zero_df.attrs["audit"]["retry_events"])
expected_user_texts = []
for row, masked in zip(tickets.to_dict("records"), masked_texts):
    if check_input_guardrails(row["text"])[0] == "eligible":
        expected_user_texts.extend([masked] * (1 + retry_counts[row["ticket_id"]]))
assert submitted_user_texts == expected_user_texts
assert len(submitted_user_texts) == 175
zero_summary = summarise_run(zero_df, zero_tokens)
print("Successful calls:", zero_df.attrs["audit"]["successful_model_calls"])
print("Transient failures recovered:", len(retry_logs["zero_shot"]))
print("Status counts:\n", zero_df["status"].value_counts().to_string())
print("All 175 attempted messages are exactly the eligible redacted texts in file order")


[RETRY] Ticket: T0002 | Retry: 0 | Wait: 0.052993s
[RETRY] Ticket: T0116 | Retry: 0 | Wait: 0.045313s
[RETRY] Ticket: T0153 | Retry: 0 | Wait: 0.048438s
[RETRY] Ticket: T0039 | Retry: 0 | Wait: 0.047790s
[RETRY] Ticket: T0152 | Retry: 0 | Wait: 0.054206s
[RETRY] Ticket: T0043 | Retry: 0 | Wait: 0.053459s
[RETRY] Ticket: T0111 | Retry: 0 | Wait: 0.056152s
[RETRY] Ticket: T0155 | Retry: 0 | Wait: 0.046087s
[RETRY] Ticket: T0100 | Retry: 0 | Wait: 0.050274s
[RETRY] Ticket: T0068 | Retry: 0 | Wait: 0.045372s
[RETRY] Ticket: T0135 | Retry: 0 | Wait: 0.047733s
[RETRY] Ticket: T0009 | Retry: 0 | Wait: 0.051317s
[RETRY] Ticket: T0067 | Retry: 0 | Wait: 0.045332s
[RETRY] Ticket: T0106 | Retry: 0 | Wait: 0.047485s
[RETRY] Ticket: T0097 | Retry: 0 | Wait: 0.053124s
[RETRY] Ticket: T0003 | Retry: 0 | Wait: 0.051812s
[RETRY] Ticket: T0019 | Retry: 0 | Wait: 0.047756s
[RETRY] Ticket: T0086 | Retry: 0 | Wait: 0.052366s
[RETRY] Ticket: T0065 | Retry: 0 | Wait: 0.055118s
Successful calls: 156
Transient

## 7. Report zero-shot metrics and the confusion table

The denominator is the 147 tickets whose category was accepted by the validator. Total usage includes successful calls whose output was invalid, since they still consumed tokens. Confusion rows are true categories and columns are predicted categories.


In [8]:
zero_classified = zero_df[zero_df["status"].isin(CLASSIFIED_STATUSES)]
zero_confusion = pd.crosstab(zero_classified["true_category"], zero_classified["category"]).reindex(
    index=ALLOWED_CATEGORIES, columns=ALLOWED_CATEGORIES, fill_value=0)
print("Zero-shot classified:", zero_summary["classified"])
print("Correct:", zero_summary["correct"])
print("Accuracy:", f"{zero_summary['accuracy']:.2%}")
print("Total tokens:", zero_summary["total_tokens"])
print("Tokens per classified ticket:", f"{zero_summary['tokens_per_classified_ticket']:.2f}")
print("Confusion table (true rows / predicted columns):\n", zero_confusion.to_string())
assert zero_confusion.loc["fraud", "billing"] == 11
assert zero_confusion.loc["technical", "billing"] == 10
assert int(zero_confusion.to_numpy().sum()) == 147


Zero-shot classified: 147
Correct: 126
Accuracy: 85.71%
Total tokens: 9400
Tokens per classified ticket: 63.95
Confusion table (true rows / predicted columns):
 category       billing  delivery  technical  fraud
true_category                                     
billing             38         0          0      0
delivery             0        36          0      0
technical           10         0         29      0
fraud               11         0          0     23


## 8. Run few-shot without a token budget and compare quality with cost

Resetting the mock inside the pipeline gives each prompt the same error schedule. Using the same seed also yields the same retry jitter. This makes the two runs comparable, including the 12 chatty outputs recovered by keyword validation and the 9 invalid outputs sent to human review.


In [9]:
runs["few_shot"] = run_ticket_triage(prompt=FEW_SHOT_PROMPT, token_budget=None,
    retry_logger=logger_for("few_shot"))
few_df, few_tokens = runs["few_shot"]
assert_full_run(few_df, correct=147)
few_summary = summarise_run(few_df, few_tokens)
assert few_df.attrs["audit"]["retry_events"] == zero_df.attrs["audit"]["retry_events"]
print("Few-shot classified:", few_summary["classified"])
print("Correct:", few_summary["correct"])
print("Accuracy:", f"{few_summary['accuracy']:.2%}")
print("Total tokens:", few_summary["total_tokens"])
print("Tokens per classified ticket:", f"{few_summary['tokens_per_classified_ticket']:.2f}")
print("\n", compare_prompts(zero_summary, few_summary))


[RETRY] Ticket: T0002 | Retry: 0 | Wait: 0.052993s
[RETRY] Ticket: T0116 | Retry: 0 | Wait: 0.045313s
[RETRY] Ticket: T0153 | Retry: 0 | Wait: 0.048438s
[RETRY] Ticket: T0039 | Retry: 0 | Wait: 0.047790s
[RETRY] Ticket: T0152 | Retry: 0 | Wait: 0.054206s
[RETRY] Ticket: T0043 | Retry: 0 | Wait: 0.053459s
[RETRY] Ticket: T0111 | Retry: 0 | Wait: 0.056152s
[RETRY] Ticket: T0155 | Retry: 0 | Wait: 0.046087s
[RETRY] Ticket: T0100 | Retry: 0 | Wait: 0.050274s
[RETRY] Ticket: T0068 | Retry: 0 | Wait: 0.045372s
[RETRY] Ticket: T0135 | Retry: 0 | Wait: 0.047733s
[RETRY] Ticket: T0009 | Retry: 0 | Wait: 0.051317s
[RETRY] Ticket: T0067 | Retry: 0 | Wait: 0.045332s
[RETRY] Ticket: T0106 | Retry: 0 | Wait: 0.047485s
[RETRY] Ticket: T0097 | Retry: 0 | Wait: 0.053124s
[RETRY] Ticket: T0003 | Retry: 0 | Wait: 0.051812s
[RETRY] Ticket: T0019 | Retry: 0 | Wait: 0.047756s
[RETRY] Ticket: T0086 | Retry: 0 | Wait: 0.052366s
[RETRY] Ticket: T0065 | Retry: 0 | Wait: 0.055118s
Few-shot classified: 147
Correc

## 9. Demonstrate the 6000-token budget

Before every attempted call, including retries, the pipeline checks whether returned usage **so far** is already at least the budget. When it is, the ticket gets `budget_exhausted`, zero tokens and human review. This is the assignment's soft ceiling: the last admitted response may cross it, then no later eligible request is sent. Failed mock attempts return no usage and therefore add no billed tokens. The final few-shot run below uses the separately required 100000-token budget.


In [10]:
TOKEN_BUDGET = 6000
runs["budget_demo"] = run_ticket_triage(prompt=ZERO_SHOT_PROMPT, token_budget=TOKEN_BUDGET,
    retry_logger=logger_for("budget_demo"))
budget_df, budget_tokens = runs["budget_demo"]
exhausted_count = int(budget_df["status"].eq("budget_exhausted").sum())
assert exhausted_count > 0
exhausted = budget_df[budget_df["status"].eq("budget_exhausted")]
assert exhausted["tokens_used"].eq(0).all()
assert exhausted["route"].eq("human_review").all()
assert exhausted["priority"].eq("NORMAL").all()
# Once returned usage reaches the ceiling, all later rows have zero model usage.
usage_so_far = budget_df["tokens_used"].cumsum().shift(fill_value=0)
assert budget_df.loc[usage_so_far.ge(TOKEN_BUDGET), "tokens_used"].eq(0).all()
budget_df[OUTPUT_COLUMNS].to_csv(OUTPUT_DIR / "triage_budget_demo.csv", index=False)
print("Budget used:", TOKEN_BUDGET)
print("Tickets budget_exhausted:", exhausted_count)
print("Billed tokens:", budget_tokens)
print("Last-call overshoot:", budget_df.attrs["audit"]["budget_overshoot_tokens"])
print("Wrote outputs/triage_budget_demo.csv")


[RETRY] Ticket: T0002 | Retry: 0 | Wait: 0.052993s
[RETRY] Ticket: T0116 | Retry: 0 | Wait: 0.045313s
[RETRY] Ticket: T0153 | Retry: 0 | Wait: 0.048438s
[RETRY] Ticket: T0039 | Retry: 0 | Wait: 0.047790s
[RETRY] Ticket: T0152 | Retry: 0 | Wait: 0.054206s
[RETRY] Ticket: T0043 | Retry: 0 | Wait: 0.053459s
[RETRY] Ticket: T0111 | Retry: 0 | Wait: 0.056152s
[RETRY] Ticket: T0155 | Retry: 0 | Wait: 0.046087s
[RETRY] Ticket: T0100 | Retry: 0 | Wait: 0.050274s
[RETRY] Ticket: T0068 | Retry: 0 | Wait: 0.045372s
[RETRY] Ticket: T0135 | Retry: 0 | Wait: 0.047733s
[RETRY] Ticket: T0009 | Retry: 0 | Wait: 0.051317s
Budget used: 6000
Tickets budget_exhausted: 56
Billed tokens: 6038
Last-call overshoot: 38
Wrote outputs/triage_budget_demo.csv


## 10. Use Part B segments and verify routing precedence

Skipped, long, invalid-output and exhausted tickets always go to `human_review`, NORMAL. Accepted fraud always goes to `security_team`, HIGH, including Premium customers. Other accepted Premium tickets go to `premium_desk`, HIGH; other tickets go to `support_queue`, NORMAL. The CSV boolean is parsed explicitly, so the string `"False"` cannot accidentally mark a customer Premium. Missing customer rows have a blank segment name and behave as non-premium.


In [11]:
segments = load_customer_segments(OUTPUT_DIR / "customer_segments.csv")
assert route_ticket("skipped_empty", "", True) == ("human_review", "NORMAL")
assert route_ticket("flagged_invalid_output", "fraud", True) == ("human_review", "NORMAL")
assert route_ticket("classified_json", "fraud", True) == ("security_team", "HIGH")
assert route_ticket("classified_json", "billing", True) == ("premium_desk", "HIGH")
assert route_ticket("classified_keyword", "technical", False) == ("support_queue", "NORMAL")
unknown = segments.get("CUSTOMER_WITHOUT_SEGMENT", {"segment_name": "", "is_premium": False})
assert unknown == {"segment_name": "", "is_premium": False}
assert route_ticket("classified_json", "delivery", unknown["is_premium"]) == ("support_queue", "NORMAL")

runs["final_few_shot"] = run_ticket_triage(prompt=FEW_SHOT_PROMPT, token_budget=100000,
    retry_logger=logger_for("final_few_shot"))
final_df, final_tokens = runs["final_few_shot"]
assert_full_run(final_df, correct=147)
assert final_df.equals(few_df)
assert final_df["ticket_id"].tolist() == tickets["ticket_id"].tolist()
for row in final_df.to_dict("records"):
    segment = segments.get(row["customer_id"], {"segment_name": "", "is_premium": False})
    assert row["segment_name"] == segment["segment_name"]
    assert (row["route"], row["priority"]) == route_ticket(row["status"], row["category"], segment["is_premium"])
    if row["status"] not in CLASSIFIED_STATUSES:
        assert row["category"] == ""
final_df[OUTPUT_COLUMNS].to_csv(OUTPUT_DIR / "triage_results.csv", index=False)
print("Final rows:", len(final_df))
print("Routing counts:\n", final_df["route"].value_counts().to_string())
print("Priority counts:\n", final_df["priority"].value_counts().to_string())
print("Wrote outputs/triage_results.csv with exactly", OUTPUT_COLUMNS)


[RETRY] Ticket: T0002 | Retry: 0 | Wait: 0.052993s
[RETRY] Ticket: T0116 | Retry: 0 | Wait: 0.045313s
[RETRY] Ticket: T0153 | Retry: 0 | Wait: 0.048438s
[RETRY] Ticket: T0039 | Retry: 0 | Wait: 0.047790s
[RETRY] Ticket: T0152 | Retry: 0 | Wait: 0.054206s
[RETRY] Ticket: T0043 | Retry: 0 | Wait: 0.053459s
[RETRY] Ticket: T0111 | Retry: 0 | Wait: 0.056152s
[RETRY] Ticket: T0155 | Retry: 0 | Wait: 0.046087s
[RETRY] Ticket: T0100 | Retry: 0 | Wait: 0.050274s
[RETRY] Ticket: T0068 | Retry: 0 | Wait: 0.045372s
[RETRY] Ticket: T0135 | Retry: 0 | Wait: 0.047733s
[RETRY] Ticket: T0009 | Retry: 0 | Wait: 0.051317s
[RETRY] Ticket: T0067 | Retry: 0 | Wait: 0.045332s
[RETRY] Ticket: T0106 | Retry: 0 | Wait: 0.047485s
[RETRY] Ticket: T0097 | Retry: 0 | Wait: 0.053124s
[RETRY] Ticket: T0003 | Retry: 0 | Wait: 0.051812s
[RETRY] Ticket: T0019 | Retry: 0 | Wait: 0.047756s
[RETRY] Ticket: T0086 | Retry: 0 | Wait: 0.052366s
[RETRY] Ticket: T0065 | Retry: 0 | Wait: 0.055118s
Final rows: 164
Routing counts:

## 11. Persist the full measured report and check reproducibility

The report records actual counts, prompt text, per-run usage, confusion tables, retry waits, budget settings, overshoot and the explanations needed for the project README. The unlimited few-shot experiment and final 100000-token run have identical output rows and usage after a reset. JSON/keyword classification is **rule validation**, not factual verification; invalid outputs and blocked requests need humans. The supplied deterministic mock is useful for an offline capstone but does not establish accuracy, cost or reliability for a real hosted model.


In [12]:
report = build_report(input_audit, runs, retry_logs)
report_path = OUTPUT_DIR / "part_c_report.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
assert few_tokens == final_tokens
for filename in ["triage_results.csv", "triage_budget_demo.csv"]:
    saved = pd.read_csv(OUTPUT_DIR / filename, keep_default_na=False)
    assert len(saved) == 164 and saved.columns.tolist() == OUTPUT_COLUMNS
assert json.loads(report_path.read_text())["final_few_shot"]["correct"] == 147
print(report["routing_note"])
print(report["budget_note"])
print("All Part C acceptance checks passed")
print("Saved reports: outputs/part_c_report.json, outputs/triage_results.csv, outputs/triage_budget_demo.csv")


classified_json and classified_keyword are rule-validated model outputs, not guarantees of truth. Rejected input, invalid output and budget-exhausted tickets go to human_review with NORMAL priority; fraud goes to security_team before Premium routing is considered. Missing segment rows are non-premium.
The 6000-token demonstration follows the specified usage-so-far rule, so its last admitted response can cross 6000. Every later eligible ticket makes no call and goes to human review.
All Part C acceptance checks passed
Saved reports: outputs/part_c_report.json, outputs/triage_results.csv, outputs/triage_budget_demo.csv
